# Learning Objectives
In this notebook, you will learn Spark Dataframe APIs.

# Question List

Solve the following questions using Spark Dataframe APIs

### Join

1. easy - https://pgexercises.com/questions/joins/simplejoin.html
2. easy - https://pgexercises.com/questions/joins/simplejoin2.html
3. easy - https://pgexercises.com/questions/joins/self2.html 
4. medium - https://pgexercises.com/questions/joins/threejoin.html (three join)
5. medium - https://pgexercises.com/questions/joins/sub.html (subquery and join)

### Aggregation

1. easy - https://pgexercises.com/questions/aggregates/count3.html Group by order by
2. easy - https://pgexercises.com/questions/aggregates/fachours.html group by order by
3. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth.html group by with condition 
4. easy - https://pgexercises.com/questions/aggregates/fachoursbymonth2.html group by multi col
5. easy - https://pgexercises.com/questions/aggregates/members1.html count distinct
6. med - https://pgexercises.com/questions/aggregates/nbooking.html group by multiple cols, join

### String & Date

1. easy - https://pgexercises.com/questions/string/concat.html format string
2. easy - https://pgexercises.com/questions/string/case.html WHERE + string function
3. easy - https://pgexercises.com/questions/string/reg.html WHERE + string function
4. easy - https://pgexercises.com/questions/string/substr.html group by, substr
5. easy - https://pgexercises.com/questions/date/series.html generate ts
6. easy - https://pgexercises.com/questions/date/bookingspermonth.html extract month from ts

In [0]:
bookings_df = spark.table("bookings")
members_df = spark.table("members")
facilities_df = spark.table("facilities")

### Question

How can you produce a list of the start times for bookings by members named 'David Farrell'?

https://pgexercises.com/questions/joins/simplejoin.html

In [0]:
# Write you solution here
# hint: you might need to re-run `0 - ETL pgexercieses CSV files` notebook to init tables

from pyspark.sql.functions import col



results = (
    bookings_df
    .join(members_df, on="memid")
    .filter(
        (col("firstname") == "David") &
        (col("surname") == "Farrell")
    )
    .select("starttime")
)


display(results)

starttime
2012-09-18T09:00:00.000Z
2012-09-18T17:30:00.000Z
2012-09-18T13:30:00.000Z
2012-09-18T20:00:00.000Z
2012-09-19T09:30:00.000Z
2012-09-19T15:00:00.000Z
2012-09-19T12:00:00.000Z
2012-09-20T15:30:00.000Z
2012-09-20T11:30:00.000Z
2012-09-20T14:00:00.000Z


### Question

How can you produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'? Return a list of start time and facility name pairings, ordered by the time.

In [0]:

result = (
    bookings_df
    .join(facilities_df, on="facid", how="inner")
    .filter(
        col("name").contains("Tennis Court") &
        (col("starttime") >= "2012-09-21 00:00:00") &
        (col("starttime") <  "2012-09-22 00:00:00")
    )
    .select("starttime", "name")
    .orderBy(col("starttime").desc())
)

display(result)

starttime,name
2012-09-21T18:00:00.000Z,Tennis Court 2
2012-09-21T17:00:00.000Z,Tennis Court 1
2012-09-21T16:00:00.000Z,Tennis Court 2
2012-09-21T15:30:00.000Z,Tennis Court 1
2012-09-21T14:00:00.000Z,Tennis Court 2
2012-09-21T13:30:00.000Z,Tennis Court 1
2012-09-21T12:00:00.000Z,Tennis Court 1
2012-09-21T11:30:00.000Z,Tennis Court 2
2012-09-21T10:00:00.000Z,Tennis Court 2
2012-09-21T09:30:00.000Z,Tennis Court 1


##Question
How can you output a list of all members, including the individual who recommended them (if any)? Ensure that results are ordered by (surname, firstname).

In [0]:
result = (members_df.alias("member").join(members_df.alias("recommended"), col("member.recommendedby") == col("recommended.memid"), how="left")
.select("member.surname", "member.firstname", col("recommended.surname").alias("recommender_surname"), col("recommended.firstname").alias("recommender_firstname"))
    .orderBy("member.surname", "member.firstname")
)

display(result)


surname,firstname,recommender_surname,recommender_firstname
Bader,Florence,Stibbons,Ponder
Baker,Anne,Stibbons,Ponder
Baker,Timothy,Farrell,Jemima
Boothe,Tim,Rownam,Tim
Butters,Gerald,Smith,Darren
Coplin,Joan,Baker,Timothy
Crumpet,Erica,Smith,Tracy
Dare,Nancy,Joplette,Janice
Farrell,David,null,null
Farrell,Jemima,null,null


## Question
How can you produce a list of all members who have used a tennis court? Include in your output the name of the court, and the name of the member formatted as a single column. Ensure no duplicate data, and order by the member name followed by the facility name.
https://pgexercises.com/questions/joins/threejoin.html

In [0]:
result = (members_df.join(bookings_df, on="memid", how="inner").join(facilities_df, on="facid", how="inner")
    .filter(col("name").contains("Tennis Court")).select("surname", "firstname", col("name").alias("Facility Name")).distinct()
    .orderBy("surname", "firstname", "Facility Name")
)

display(result)

surname,firstname,Facility Name
Bader,Florence,Tennis Court 1
Bader,Florence,Tennis Court 2
Baker,Anne,Tennis Court 1
Baker,Anne,Tennis Court 2
Baker,Timothy,Tennis Court 1
Baker,Timothy,Tennis Court 2
Boothe,Tim,Tennis Court 1
Boothe,Tim,Tennis Court 2
Butters,Gerald,Tennis Court 1
Butters,Gerald,Tennis Court 2


## Question 
How can you output a list of all members, including the individual who recommended them (if any), without using any joins? Ensure that there are no duplicates in the list, and that each firstname + surname pairing is formatted as a column and ordered.

In [0]:
from pyspark.sql.functions import col, concat_ws, create_map, lit
from itertools import chain

# Create a mapping of memid to full name
member_map_data = (
    members_df
    .select(
        col("memid"),
        concat_ws(" ", col("firstname"), col("surname")).alias("fullname")
    )
    .collect()
)

# Convert to a dictionary and then to a Spark map
member_dict = {row.memid: row.fullname for row in member_map_data}
mapping_expr = create_map([lit(x) for x in chain(*member_dict.items())])

# Use the map to look up recommender names
result = (
    members_df
    .withColumn(
        "member_fullname",
        concat_ws(" ", col("firstname"), col("surname"))
    )
    .withColumn(
        "recommender_fullname",
        mapping_expr[col("recommendedby")]
    )
    .select("member_fullname", "recommender_fullname")
    .dropDuplicates()
    .orderBy("member_fullname")
)

display(result)

member_fullname,recommender_fullname
Anna Mackenzie,Darren Smith
Anne Baker,Ponder Stibbons
Burton Tracy,null
Charles Owen,Darren Smith
Darren Smith,null
David Farrell,null
David Jones,Janice Joplette
David Pinker,Jemima Farrell
Douglas Jones,David Jones
Erica Crumpet,Tracy Smith


## Question
Produce a count of the number of recommendations each member has made. Order by member ID.


In [0]:
from pyspark.sql.functions import col, count


result = (members_df.filter(col("recommendedby").isNotNull()).groupBy("recommendedby").agg(count("*").alias("count")).orderBy(col("recommendedby").asc(), col("recommendedby").asc())
          .withColumnRenamed("recommendedby", "recommender_id"))
display(result)

recommender_id,count
1,5
2,3
3,1
4,2
5,1
6,1
9,2
11,1
13,2
15,1


## Question
Produce a list of the total number of slots booked per facility. For now, just produce an output table consisting of facility id and slots, sorted by facility id.

In [0]:
from pyspark.sql.functions import col, sum


result = (bookings_df.join(facilities_df, on="facid", how="inner").groupBy("facid").agg(sum("slots").alias("total_slots"))).orderBy("facid", ascending=True)
display(result)

facid,total_slots
0,1320
1,1278
2,1209
3,830
4,1404
5,228
6,1104
7,908
8,911


Produce a list of the total number of slots booked per facility in the month of September 2012. Produce an output table consisting of facility id and slots, sorted by the number of slots.

In [0]:
from pyspark.sql.functions import col, sum


result = (bookings_df.join(facilities_df, on="facid", how="inner").filter(col("starttime").between("2012-09-01", "2012-09-30")).groupBy("facid").agg(sum("slots").alias("total_slots"))).orderBy("facid", ascending=True)
display(result)

facid,total_slots
0,567
1,567
2,546
3,410
4,628
5,118
6,522
7,412
8,453


## Question 
Produce a list of the total number of slots booked per facility per month in the year of 2012. Produce an output table consisting of facility id and slots, sorted by the id and month.

In [0]:
from pyspark.sql.functions import col, sum, year, month

result = (
    bookings_df
    .join(facilities_df, on="facid", how="inner")
    .filter(year(col("starttime")) == 2012)
    .groupBy("facid", month(col("starttime")).alias("month"))
    .agg(sum("slots").alias("total_slots"))
    .orderBy("facid", "month")
)

display(result)

facid,month,total_slots
0,7,270
0,8,459
0,9,591
1,7,207
1,8,483
1,9,588
2,7,180
2,8,459
2,9,570
3,7,104


## Question
Find the count of members who have made at least one booking

In [0]:
result = (bookings_df.select("memid").distinct().count())
display(result)

30

## Question
Produce a list of each member name, id, and their first booking after September 1st 2012. Order by member ID.


In [0]:
from pyspark.sql.functions import col, min
members_df = members_df.withColumn(
    "fullname",
    concat_ws(" ", col("firstname"), col("surname"))
)

result = (members_df.join(bookings_df, on="memid", how="inner").filter(col("starttime") >= "2012-09-01 00:00:00").groupBy("fullname").agg(min("starttime")).orderBy("fullname").withColumnRenamed("min(starttime)", "earliest_starttime"))
display(result)


fullname,earliest_starttime
Anna Mackenzie,2012-09-01T08:30:00.000Z
Anne Baker,2012-09-01T14:30:00.000Z
Burton Tracy,2012-09-01T15:00:00.000Z
Charles Owen,2012-09-01T11:00:00.000Z
Darren Smith,2012-09-01T09:00:00.000Z
David Farrell,2012-09-18T09:00:00.000Z
David Jones,2012-09-01T09:30:00.000Z
David Pinker,2012-09-01T08:30:00.000Z
Douglas Jones,2012-09-08T13:00:00.000Z
Erica Crumpet,2012-09-27T11:30:00.000Z


## Question
Output the names of all members, formatted as 'Surname, Firstname'



In [0]:


result = members_df.select(concat_ws(", ", col("surname"), col("firstname")).alias("name")).distinct().orderBy("name")

display(result)

name
"Bader, Florence"
"Baker, Anne"
"Baker, Timothy"
"Boothe, Tim"
"Butters, Gerald"
"Coplin, Joan"
"Crumpet, Erica"
"Dare, Nancy"
"Farrell, David"
"Farrell, Jemima"


## Question
Perform a case-insensitive search to find all facilities whose name begins with 'tennis'. Retrieve all columns.


In [0]:
from pyspark.sql.functions import col

result = facilities_df.filter(col("name").ilike("tennis%"))

display(result)

facid,name,membercost,guestcost,initialoutlay,monthlymaintenance
0,Tennis Court 1,5,25,10000,200
1,Tennis Court 2,5,25,8000,200


## Question
You've noticed that the club's member table has telephone numbers with very inconsistent formatting. You'd like to find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.


In [0]:
from pyspark.sql.functions import col

result = (
    members_df
    .filter(col("telephone").rlike(r"\("))
    .select("memid", "telephone")
    .orderBy("memid")
)

display(result)

memid,telephone
0,(000) 000-0000
3,(844) 693-0723
4,(833) 942-4710
5,(844) 078-4130
6,(822) 354-9973
7,(833) 776-4001
8,(811) 433-2547
9,(833) 160-3900
10,(855) 542-5251
11,(844) 536-8036


You'd like to produce a count of how many members you have whose surname starts with each letter of the alphabet. Sort by the letter, and don't worry about printing out a letter if the count is 0.

In [0]:
from pyspark.sql.functions import col, substring, upper, count

result = (
    members_df
    .withColumn("surname_initial", upper(substring(col("surname"), 1, 1)))
    .groupBy("surname_initial")
    .agg(count("*").alias("member_count"))
    .orderBy("surname_initial")
)

display(result)

surname_initial,member_count
B,5
C,2
D,1
F,2
G,2
H,1
J,3
M,1
O,1
P,2


## Question
Produce a list of all the dates in October 2012. They can be output as a timestamp (with time set to midnight) or a date.


In [0]:
from pyspark.sql.functions import sequence, to_date, explode, lit

result = (
    spark
    .range(1)
    .select(
        sequence(
            to_date(lit("2012-10-01")),
            to_date(lit("2012-10-31"))
        ).alias("date_seq")
    )
    .select(explode("date_seq").alias("date"))
)

display(result)

date
2012-10-01
2012-10-02
2012-10-03
2012-10-04
2012-10-05
2012-10-06
2012-10-07
2012-10-08
2012-10-09
2012-10-10


Return a count of bookings for each month, sorted by month


In [0]:
from pyspark.sql.functions import month, count

result = (
    bookings_df
    .groupBy(month("starttime").alias("month"))
    .agg(count("*").alias("booking_count"))
    .orderBy("month")
)

display(result)

month,booking_count
1,1
7,658
8,1472
9,1913
